# 01 · Preprocessing and EDA
**Owner:** Chaitanya Sai Meka

Load the three official Corr2Cause splits, clean them, remove leakage, subsample train and explore the data.

* **train**: used to fit everything
* **validation**: used to pick the model
* **test**: used once at the end

In [1]:
import hashlib
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from huggingface_hub import hf_hub_download

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

SEED = 42
sns.set_theme(style="whitegrid")

## 1. Load the data
Hugging Face already gives us train, validation (`dev.csv`) and test, so we keep those splits as they are.

In [2]:
files = {"train": "train.csv", "validation": "dev.csv", "test": "test.csv"}
raw = {}
for split, name in files.items():
    path = hf_hub_download("causal-nlp/corr2cause", name, repo_type="dataset", local_dir=RAW)
    raw[split] = pd.read_csv(path)
    print(split, raw[split].shape)

raw["train"].head()

train (205734, 4)


validation (1076, 4)


test (1162, 4)


,input,label,num_variables,template
0,Premise: Suppose there is a closed system of 4...,0,4,has_collider
1,Premise: Suppose there is a closed system of 4...,0,4,non-child descendant
2,Premise: Suppose there is a closed system of 4...,0,4,child
3,Premise: Suppose there is a closed system of 4...,0,4,non-parent ancestor
4,Premise: Suppose there is a closed system of 4...,1,4,parent


## 2. Data quality checks

In [3]:
pd.DataFrame({
    split: {
        "missing values": df.isna().sum().sum(),
        "duplicate rows": df.duplicated().sum(),
        "labels": sorted(df["label"].unique()),
        "rows without one 'Hypothesis:'": (df["input"].str.count("Hypothesis:") != 1).sum(),
    }
    for split, df in raw.items()
}).T

,missing values,duplicate rows,labels,rows without one 'Hypothesis:'
train,0,0,"[0, 1]",0
validation,0,0,"[0, 1]",0
test,0,0,"[0, 1]",0


No missing values and no duplicates, so there is nothing to impute or drop.
The only cleaning needed is splitting `input` into premise and hypothesis and tidying the whitespace.

In [4]:
def clean(df, split):
    text = df["input"].str.replace(r"\s+", " ", regex=True).str.strip()
    parts = text.str.split("Hypothesis:", n=1)
    out = pd.DataFrame({
        "premise": parts.str[0].str.replace("Premise:", "", n=1).str.strip(),
        "hypothesis": parts.str[1].str.strip(),
        "label": df["label"],
        "num_variables": df["num_variables"],
        "template": df["template"].str.strip(),
    })
    # a short id per premise, so we can group rows that share the same premise
    # (the "p" prefix stops pandas from reading the id back as a number)
    out.insert(0, "premise_id", out["premise"].map(lambda p: "p" + hashlib.md5(p.encode()).hexdigest()[:8]))
    return out


data = {split: clean(df, split) for split, df in raw.items()}
data["train"].head(3)

,premise_id,premise,hypothesis,label,num_variables,template
0,p2497edc2,Suppose there is a closed system of 4 variable...,"There exists at least one collider (i.e., comm...",0,4,has_collider
1,pc7f4a671,Suppose there is a closed system of 4 variable...,"C is a cause for B, but not a direct one.",0,4,non-child descendant
2,pc7f4a671,Suppose there is a closed system of 4 variable...,C directly causes B.,0,4,child


In [5]:
# num_variables should match the number written in the premise itself
for split, df in data.items():
    stated = df["premise"].str.extract(r"closed system of (\d+) variables")[0].astype(int)
    print(split, "mismatches:", (stated != df["num_variables"]).sum())

train mismatches: 0
validation mismatches: 0
test mismatches: 0


## 3. Leakage check
The same premise is reused with many hypotheses. If a premise is in both train and test, the model has already
seen it, so we check for overlap between the splits.

In [6]:
ids = {split: set(df["premise_id"]) for split, df in data.items()}
pd.DataFrame([[len(ids[a] & ids[b]) for b in ids] for a in ids], index=ids, columns=ids)

,train,validation,test
train,2338,2,0
validation,2,22,3
test,0,3,21


A few premises are shared. We keep test untouched and remove the shared premises from validation and train.

In [7]:
data["validation"] = data["validation"][~data["validation"]["premise_id"].isin(ids["test"])]
held_out = set(data["validation"]["premise_id"]) | ids["test"]
data["train"] = data["train"][~data["train"]["premise_id"].isin(held_out)]

ids = {split: set(df["premise_id"]) for split, df in data.items()}
assert not (ids["train"] & ids["validation"]) and not (ids["train"] & ids["test"]) and not (ids["validation"] & ids["test"])
print("no premise is shared between splits")

no premise is shared between splits


## 4. Subsample train
Train has ~200k rows, which is more than we need. We sample 400 whole premises (not single rows) with a fixed seed,
so every premise keeps all of its hypotheses.

In [8]:
chosen = data["train"]["premise_id"].drop_duplicates().sample(n=400, random_state=SEED)
train = data["train"][data["train"]["premise_id"].isin(chosen)].reset_index(drop=True)
validation = data["validation"].reset_index(drop=True)
test = data["test"].reset_index(drop=True)

for name, df in [("train", train), ("validation", validation), ("test", test)]:
    print(f"{name:<11} {len(df):>6} rows, {df['premise_id'].nunique():>3} premises")

train        35340 rows, 400 premises
validation    1026 rows,  19 premises
test          1162 rows,  21 premises


In [9]:
# the subsample should look like the full train split
pd.DataFrame({"full train": data["train"]["label"].mean(), "subsample": train["label"].mean()},
             index=["share of label 1"]).round(3)

,full train,subsample
share of label 1,0.185,0.185
